# Component Tests — Real-Time Fraud Detection Pipeline

Test each component individually to verify the pipeline is healthy.

In [10]:
# --- Session setup (local or Container Runtime) ---
from snowflake.snowpark import Session

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    session = Session.builder.configs({"connection_name": "ereitberger-demo-account-aws-eu-frankfurt"}).create()

session.sql("USE ROLE ACCOUNTADMIN").collect()
session.sql("USE WAREHOUSE COMPUTE_WH").collect()
session.sql("USE DATABASE FRAUD_ML").collect()
session.sql("USE SCHEMA FEATURE_STORE").collect()
print("Session ready:", session.get_current_role())

Session ready: "ACCOUNTADMIN"


## 1. Managed Mirror — is the source table populated?

In [2]:
session.sql("SELECT COUNT(*) AS ROW_COUNT FROM FRAUD_DEMO_OLTP.PUBLIC.TRANSACTIONS").show()
session.sql("SELECT * FROM FRAUD_DEMO_OLTP.PUBLIC.TRANSACTIONS ORDER BY CREATED_AT DESC LIMIT 3").show()

---------------
|"ROW_COUNT"  |
---------------
|228          |
---------------

---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"ID"  |"CARD_TYPE"  |"TRANSACTION_TYPE"  |"CARDHOLDER_NAME"  |"CARD_NUMBER"     |"EXPIRY"  |"CVC"  |"BILLING_LOCATION"  |"CREATED_AT"                      |"AMOUNT"  |"MERCHANT_CATEGORY"  |"LATITUDE"  |"LONGITUDE"  |"FRAUD_FLAG"  |"FRAUD_REASON"        |"ANOMALY_SCORE"  |
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|243   |amex         |pos                 |Bob Novak          |371528664855286   |04/27     |2594

## 2. Dynamic Table — CARD_FEATURES (one row per card)

In [3]:
session.sql("SELECT COUNT(*) AS CARDS FROM FRAUD_ML.FEATURE_STORE.CARD_FEATURES").show()
session.sql("SELECT * FROM FRAUD_ML.FEATURE_STORE.CARD_FEATURES LIMIT 5").show()

-----------
|"CARDS"  |
-----------
|46       |
-----------

----------------------------------------------------------------------------------------------------
|"CARD_NUMBER"    |"HOME_LAT"  |"HOME_LON"  |"LAST_LAT"  |"LAST_LON"  |"AVG_AMOUNT"  |"TXN_COUNT"  |
----------------------------------------------------------------------------------------------------
|342439278037808  |41.8781     |-87.6298    |41.8781     |-87.6298    |260.02714286  |7            |
|343135561393975  |52.52       |13.405      |40.7128     |-74.006     |229.99000000  |4            |
|343318350794267  |40.7128     |-74.006     |40.7128     |-74.006     |557.32000000  |4            |
|344825084204084  |25.2048     |55.2708     |25.2048     |55.2708     |631.99666667  |6            |
|345003818505268  |51.5074     |-0.1278     |51.5074     |-0.1278     |169.45142857  |7            |
----------------------------------------------------------------------------------------------------



## 3. Online Feature Store — retrieve features by CARD_NUMBER

In [11]:
import os
from snowflake.ml.feature_store import FeatureStore, CreationMode
from snowflake.ml.feature_store.feature_view import StoreType

os.environ["SNOWFLAKE_PAT"] = session.connection.rest.token

fs = FeatureStore(
    session=session,
    database="FRAUD_ML",
    name="FEATURE_STORE",
    default_warehouse="COMPUTE_WH",
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)

card_fv = fs.get_feature_view("CARD_FRAUD_FEATURES", "V1")
sample_card = session.sql("SELECT CARD_NUMBER FROM FRAUD_ML.FEATURE_STORE.CARD_FEATURES LIMIT 1").collect()[0]["CARD_NUMBER"]
print(f"FeatureView: {card_fv.name}, online: {card_fv.online_config.store_type}, lag: {card_fv.online_config.target_lag}")

# Pick a real card number
sample_card = session.sql("SELECT CARD_NUMBER FROM FRAUD_ML.FEATURE_STORE.CARD_FEATURES LIMIT 1").collect()[0]["CARD_NUMBER"]
print(f"\nLooking up card: {sample_card}")

result = fs.read_feature_view(card_fv, keys=[[sample_card]], store_type=StoreType.ONLINE)
print("Online store response:")
result

FeatureView: CARD_FRAUD_FEATURES, online: OnlineStoreType.POSTGRES, lag: 10 seconds

Looking up card: 5612987554485883


Online Service HTTP error: status=500 request_id=62e2432b-62c2-4d06-a121-8154e2ac4c2e body='{"responseType":"FAULT","requestId":"62e2432b-62c2-4d06-a121-8154e2ac4c2e","detail":""}'


RuntimeError: (1500) Online Service error (HTTP 500): {"responseType":"FAULT","requestId":"62e2432b-62c2-4d06-a121-8154e2ac4c2e","detail":""} [request_id: 62e2432b-62c2-4d06-a121-8154e2ac4c2e]

## 4. Model Registry — check model exists and list functions

In [9]:
from snowflake.ml.registry import Registry

session.sql("USE DATABASE FRAUD_ML").collect()
session.sql("USE SCHEMA FEATURE_STORE").collect()

reg = Registry(session=session, database_name="FRAUD_ML", schema_name="FEATURE_STORE")
mv = reg.get_model("FRAUD_XGBOOST").version("V1")
print("Model:", mv.model_name, mv.version_name)
print("Functions:", mv.show_functions())

/Users/ereitberger/dev_projects/credit_card_fraud_detection_demo/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model: FRAUD_XGBOOST V1
Functions: [{'name': 'PREDICT', 'target_method': 'predict', 'target_method_function_type': 'FUNCTION', 'signature': ModelSignature(
    inputs=[
        FeatureSpec(dtype=DataType.DOUBLE, name='AMOUNT', nullable=True),
        FeatureSpec(dtype=DataType.INT8, name='MERCHANT_CATEGORY', nullable=True),
        FeatureSpec(dtype=DataType.DOUBLE, name='LATITUDE', nullable=True),
        FeatureSpec(dtype=DataType.DOUBLE, name='LONGITUDE', nullable=True),
        FeatureSpec(dtype=DataType.DOUBLE, name='HOME_LAT', nullable=True),
        FeatureSpec(dtype=DataType.DOUBLE, name='HOME_LON', nullable=True),
        FeatureSpec(dtype=DataType.DOUBLE, name='LAST_LAT', nullable=True),
        FeatureSpec(dtype=DataType.DOUBLE, name='LAST_LON', nullable=True),
        FeatureSpec(dtype=DataType.DOUBLE, name='AVG_AMOUNT', nullable=True),
        FeatureSpec(dtype=DataType.INT8, name='TXN_COUNT', nullable=True)
    ],
    outputs=[
        FeatureSpec(dtype=DataType.INT64, na

## 5. SPCS Inference Service — call /predict directly

In [ ]:
import httpx

PREDICT_URL = "https://mn5jie-sfseeurope-ereitberger-aws2.snowflakecomputing.app/predict"
PAT = os.environ.get("FRAUD_PREDICT_TOKEN") or session.connection.rest.token

# Test with a known card + a suspicious charge (high amount, far from home)
payload = {
    "dataframe_split": {
        "index": [0],
        "columns": ["CARD_NUMBER", "AMOUNT", "MERCHANT_CATEGORY", "LATITUDE", "LONGITUDE"],
        "data": [[sample_card, 1999.99, 5, 40.71, -74.01]]  # Jewelry, NYC
    }
}

resp = httpx.post(
    PREDICT_URL,
    headers={"Authorization": f'Snowflake Token="{PAT}"', "Content-Type": "application/json"},
    json=payload,
    timeout=30,
)
print(f"Status: {resp.status_code}")
print(f"Response: {resp.json()}")

# Interpret
data = resp.json().get("data", [[None, None]])[0]
score = data[1].get("output_feature_0") if isinstance(data[1], dict) else data[1]
print(f"\nFraud class: {score} ({'FRAUD' if score == 1 else 'LEGIT'})")

## 6. End-to-end latency test

In [ ]:
import time

# Run 5 predictions and measure latency
latencies = []
for i in range(5):
    t0 = time.time()
    r = httpx.post(
        PREDICT_URL,
        headers={"Authorization": f'Snowflake Token="{PAT}"', "Content-Type": "application/json"},
        json=payload,
        timeout=30,
    )
    latencies.append(time.time() - t0)
    assert r.status_code == 200, f"Failed: {r.text}"

print(f"Latencies (5 calls): {[f'{l:.2f}s' for l in latencies]}")
print(f"Avg: {sum(latencies)/len(latencies):.2f}s | Min: {min(latencies):.2f}s | Max: {max(latencies):.2f}s")